In [ ]:
import trimesh

In [ ]:

import os 
root=os.path.abspath(os.path.join(os.getcwd(), "..", "..", "rsc", "ycb_crossDex"))
dirs = os.listdir(root)
for dir in dirs:
    if os.path.isdir(os.path.join(root,dir)):
        aff_mesh_name_mixed_train=f"{root}/{dir}/top_watertight_tiny.obj"
        print(trimesh.load_mesh(aff_mesh_name_mixed_train))



In [ ]:
import os
import trimesh

# Source directory containing high-resolution meshes
src_root =  root

# Optional: save into a separate directory to preserve the source data
# dst_root = os.path.join(os.path.dirname(root), "DEXYCB_ObjForRetarget_simplified")
# os.makedirs(dst_root, exist_ok=True)

# Target face count, following the approximate mixed_train scale
TARGET_FACE_COUNT = 500  # Adjust within 300–800 when tuning mesh quality.

dirs = os.listdir(src_root)
for dir_name in sorted(dirs):  # sorted for reproducibility
    dir_path = os.path.join(src_root, dir_name)
    if not os.path.isdir(dir_path):
        continue

    mesh_path = os.path.join(dir_path, "top_watertight_tiny.obj")
    if not os.path.exists(mesh_path):
        print(f"⚠️  Missing mesh: {mesh_path}")
        continue

    try:
        mesh = trimesh.load_mesh(mesh_path)
        print(f"Original: {dir_name} → vertices={mesh.vertices.shape[0]}, faces={mesh.faces.shape[0]}")

        # 如果已经足够小，跳过简化
        if mesh.faces.shape[0] <= TARGET_FACE_COUNT:
            print(f"   → 已满足目标，跳过简化")
            continue

        # 简化网格（保留 watertight 性质？不一定，但通常够用）
        simplified_mesh = mesh.simplify_quadric_decimation(face_count=TARGET_FACE_COUNT)

        # 保存回原路径（或另存）
        simplified_mesh.export(mesh_path)  # 覆盖原文件
        # 如果用 dst_root，则：
        # os.makedirs(os.path.join(dst_root, dir_name), exist_ok=True)
        # simplified_mesh.export(os.path.join(dst_root, dir_name, "top_watertight_tiny.obj"))

        print(f"简化后: {dir_name} → vertices={simplified_mesh.vertices.shape[0]}, faces={simplified_mesh.faces.shape[0]}")

    except Exception as e:
        print(f"❌ 处理 {dir_name} 时出错: {e}")